# Probability and Measurement (No Quantum Yet)

Quantum measurement is probabilistic. Before we make it quantum we will get very comfortable with classical probability — distributions, sampling, expectation values, and the law of large numbers.

**Objectives:**
- Define and visualize discrete probability distributions
- Sample from a distribution and watch empirical frequencies converge
- Compute expectation values
- Preview the Born rule: quantum measurement is **just** sampling from a   distribution computed from a state vector

**Reference:** See [`../GUIDE.md`](../GUIDE.md).

<!-- browser-runnable -->


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. A probability distribution is just a list of numbers

Two rules:

1. Every entry is `>= 0`.
2. They sum to `1`.

That is the whole definition.

In [ ]:
outcomes = ['heads', 'tails']
probs = np.array([0.7, 0.3])           # a biased coin

assert np.all(probs >= 0)
assert np.isclose(probs.sum(), 1.0)

plt.bar(outcomes, probs, color=['#ff9900', '#146eb4'])
plt.ylabel('Probability')
plt.title('Biased coin distribution')
plt.ylim(0, 1)
plt.show()

## 2. Sampling: turning a distribution into actual outcomes

If you flip the biased coin once, you might get either result. If you flip it many times and tally, the fraction of heads will get close to 0.7.

In [ ]:
def flip(n_shots, probs):
    return np.random.choice(len(probs), size=n_shots, p=probs)

for n in [10, 100, 1000, 100_000]:
    samples = flip(n, probs)
    p_heads = np.mean(samples == 0)
    print(f'{n:>7} shots  ->  empirical P(heads) = {p_heads:.4f}')

Empirical frequency converges to the true probability as the number of shots grows. This is the **law of large numbers**. In quantum computing, the "shots" argument you pass to a simulator or QPU is exactly this idea.

## 3. Expectation value

If each outcome `i` has a numerical value `v_i` and probability `p_i`, the **expectation value** is

$$\langle V \rangle = \sum_i p_i \, v_i$$

It is just a weighted average.

In [ ]:
# Suppose heads pays $1 and tails costs $0.50. Expected payoff per flip?
values = np.array([1.0, -0.5])
ev = (probs * values).sum()
print(f'Expected payoff = ${ev:.3f}')

# Confirm empirically with many flips
samples = flip(100_000, probs)
payoffs = values[samples]
print(f'Empirical average over 100k flips = ${payoffs.mean():.3f}')

## 4. Visualizing convergence

Watch the running average settle as samples accumulate.

In [ ]:
samples = flip(2000, probs)
running_heads_frac = np.cumsum(samples == 0) / np.arange(1, len(samples) + 1)

plt.plot(running_heads_frac, color='#ff9900', linewidth=1)
plt.axhline(0.7, color='gray', linestyle='--', label='true P(heads)')
plt.xlabel('Number of flips')
plt.ylabel('Empirical P(heads)')
plt.title('Empirical frequency converging to true probability')
plt.legend()
plt.show()

## 5. A first taste of the Born rule

Here is a tiny preview of where this is heading. In quantum mechanics, a state of a single qubit is a 2-vector

$$|\psi\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}, \qquad |\alpha|^2 + |\beta|^2 = 1.$$

When you **measure** it in the computational basis, you get outcome `0` with probability $|\alpha|^2$ and outcome `1` with probability $|\beta|^2$.

That is the entire "Born rule". Quantum measurement is sampling from the distribution `[|alpha|^2, |beta|^2]`.

In [ ]:
# Example: a single qubit in state (sqrt(0.7), sqrt(0.3))
psi = np.array([np.sqrt(0.7), np.sqrt(0.3)])
print('norm =', np.linalg.norm(psi))     # 1

probs_quantum = np.abs(psi) ** 2
print('measurement distribution:', probs_quantum)

# Simulate 'measuring' this qubit 1000 times
outcomes = np.random.choice([0, 1], size=1000, p=probs_quantum)
print('empirical fraction of 0s:', np.mean(outcomes == 0))

Notice that the quantum part is just one line: `np.abs(psi) ** 2`. Everything else is ordinary probability. We will build out exactly how `psi` is constructed in the next two notebooks.

## 6. Self-check

In [ ]:
# Q1: Define a distribution over four outcomes [A, B, C, D] = [0.1, 0.2, 0.3, 0.4].
# Sample 50_000 times and verify each empirical frequency is within 0.01 of the true one.
# TODO


In [ ]:
# Q2: A die has faces 1..6, all equally likely. What is the expectation value of the face?
# Compute it analytically and confirm by sampling.
# TODO


In [ ]:
# Q3: Given the qubit state psi = [1/sqrt(3), sqrt(2)/sqrt(3)],
# what is P(measure 0)? What is P(measure 1)?
# TODO


### Solutions

In [ ]:
# --- Q1 ---
p = np.array([0.1, 0.2, 0.3, 0.4])
samples = np.random.choice(4, size=50_000, p=p)
emp = np.bincount(samples, minlength=4) / 50_000
print('true     :', p)
print('empirical:', emp)
print('all within 0.01?', np.all(np.abs(emp - p) < 0.01))

# --- Q2 ---
faces = np.arange(1, 7)
p = np.full(6, 1/6)
print('EV (analytic) =', (faces * p).sum())                   # 3.5
print('EV (sampled)  =', np.random.choice(faces, 100_000).mean())

# --- Q3 ---
psi = np.array([1/np.sqrt(3), np.sqrt(2)/np.sqrt(3)])
print('P(0) =', abs(psi[0])**2)        # 1/3
print('P(1) =', abs(psi[1])**2)        # 2/3

**You finished notebook 3.** Move on to [`04-what-is-a-qubit.ipynb`](04-what-is-a-qubit.ipynb).